LLMGATEWAYS


In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("GEMINI_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ✅
Anthropic key loaded:  ✅
Groq key loaded:       ✅


The Simplest LiteLLM Example — Unified API

In [19]:
from litellm import completion

response_groq = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}],
)

print("groq:",response_groq.choices[0].message.content)
print("*"*8)
response_gemini = completion(
    model="gemini/gemini-3.8-flash",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}],
)
print("gemini:",response_gemini.choices[0].message.content)


groq: Retrieval‑Augmented Generation (RAG) is a technique that combines a language model with an external knowledge base, first retrieving relevant documents and then conditioning the model’s generation on that retrieved information to produce more accurate and up‑to‑date responses.
********
gemini: **Retrieval-Augmented Generation (RAG)** is an AI framework that improves the accuracy of a language model by retrieving relevant information from external databases before generating an answer.


In [20]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/openai/gpt-oss-120b"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-3.8-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : Retrieval‑Augmented Generation (RAG) is a method that combines a large language 
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : **Retrieval-Augmented Generation (RAG)** is an AI technique that enhances a lang


Automatic Fallbacks — When OpenAI Goes Down

In [23]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",
        "gemini/gemini-3.8-flash"
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

17:03:27 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "d:\agentic ai\LLMGateways\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2981, in async_completion
    response: Final = await client.post(
                      ^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "d:\agentic ai\LLMGateways\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 344, in async_wrapper
    result: Final = await func(*args, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\agentic ai\LLMGateways\.ven

Response: An **LLM Gateway** (Large Language Model Gateway) is a specialized API proxy that sits between your applications and the various AI model providers you use (like OpenAI, Anthropic, Cohere, or self-hos ...

Which model actually answered? gemini-3.8-flash


In [25]:
from litellm import completion

# Force the primary to fail by using a fake model name
# Then watch the fallback chain rescue the call
response = completion(
    model="openai/fake-nonexistent-model-9999",     # 👈 will fail intentionally
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",                              # 1st backup: real OpenAI model
        "groq/openai/gpt-oss-120b"              # 2nd backup: Groq
    ]
)

print("✅ App still got a response, even though the primary failed!")
print(f"\n🤖 Model that actually answered: {response.model}")
print(f"\n📝 Response: {response.choices[0].message.content[:200]}...")

17:04:42 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model openai/fake-nonexistent-model-9999: litellm.NotFoundError: OpenAIException - The model `fake-nonexistent-model-9999` does not exist or you do not have access to it.
Traceback (most recent call last):
  File "d:\agentic ai\LLMGateways\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 985, in acompletion
    headers, response = await self.make_openai_chat_completion_request(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "d:\agentic ai\LLMGateways\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 344, in async_wrapper
    result: Final = await func(*args, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\agentic ai\LLMGateways\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 539, in make_openai_chat_completion_request
    raise e
  File "d:\agentic ai\LLMGateways\.venv\Lib\site

✅ App still got a response, even though the primary failed!

🤖 Model that actually answered: openai/gpt-oss-120b

📝 Response: ## TL;DR  
An **LLM (Large Language Model) Gateway** is a dedicated “front‑door” service that sits between your applications (or users) and one or more LLM providers (OpenAI, Anthropic, Cohere, etc.)....


Cost Tracking — Know Where Your Money Goes

In [27]:
from litellm import completion, completion_cost

response = completion(
    model="gemini/gemini-3.8-flash",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:     Born of human thought,
Silicon begins to dream,
Future wakes in code.

Input tokens:  8
Output tokens: 560
Cost:         $0.00210600


Caching — Don't Pay Twice for the Same Question

In [28]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("✅ LiteLLM state reset — ready for clean caching demo")

✅ LiteLLM state reset — ready for clean caching demo


In [33]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="gemini/gemini-3.8-flash",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"❄️  First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

❄️  First call (API):   8.56s — LLM stands for Large Language Model.
⚡ Second call (cache): 0.8260s — LLM stands for **Large Language Model**.

🚀 Speedup: 10.4x faster, and ZERO cost on the second call!


Smart Routing — The Right Model for the Right Job

In [35]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              # 👈 alias kept
        "litellm_params": {
            "model": "gemini/gemini-3.8-flash",                                      # 👈 mapped to OpenAI instead
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]
router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  AI is reshaping software development and usage in several key ways:

1. **Automation of Coding Tasks** – Code‑completion tools, generative models, and

🧠 Smart/coding (GPT-4o):
 The most Pythonic and efficient way to reverse a string is by using **slicing**.

Here is the function:

```python
def reverse_string(s: str) -> str:
    """Returns the reversed version of the input string."""
    return s[::-1]


# Example usage:
original = "Hello, World!"
reversed_text = reverse_s


 Load Balancing Across Multiple API Keys

In [38]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              # 👈 alias kept
        "litellm_params": {
            "model": "gemini/gemini-3.8-flash",                                      # 👈 mapped to OpenAI instead
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]
router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(3):
    r = router.completion(
        model="gemini/gemini-3.8-flash",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        ebcc14884248aa6c9afd4274587cbb1b821b8aa74353d50b81b81c0597ff8875  3710 ms   Hello! How can I help you with requ
#2        ebcc14884248aa6c9afd4274587cbb1b821b8aa74353d50b81b81c0597ff8875  6126 ms   Hello! How can I help you with your
#3        ebcc14884248aa6c9afd4274587cbb1b821b8aa74353d50b81b81c0597ff8875  8952 ms   Hello! Could you please provide 3 t


In [40]:
import os
from litellm import Router
from collections import Counter
model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              # 👈 alias kept
        "litellm_params": {
            "model": "gemini/gemini-3.8-flash",                                      # 👈 mapped to OpenAI instead
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   # 👈 the magic
)


hits = Counter()
for i in range(8):
    r = router.completion(
        model="gemini/gemini-3.8-flash",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → ebcc14884248aa6c9afd4274587cbb1b821b8aa74353d50b81b81c0597ff8875
Request 2 → ebcc14884248aa6c9afd4274587cbb1b821b8aa74353d50b81b81c0597ff8875


RateLimitError: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 19h12m12.446024128s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quotaMetric": "ge

In [41]:
import os
from litellm import Router
import time

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-3.8-flash",
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🔵 geminiai"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/openai/gpt-oss-120b",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq "}},
    
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"   # 👈 picks the fastest
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

Req   Deployment                      Latency   
--------------------------------------------------
#1    🟢 Groq                            1573 ms
#2    🟢 Groq                             786 ms
#3    🟢 Groq                             546 ms
#4    🟢 Groq                             328 ms
#5    🟢 Groq                             634 ms
#6    🟢 Groq                             649 ms
#7    🟢 Groq                             534 ms
#8    🟢 Groq                             736 ms
#9    🟢 Groq                            1122 ms
#10   🟢 Groq                             799 ms


In [42]:
router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"   # 👈 valid strategy
)

for i in range(5):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

Request 1 → 🟢 Groq 
Request 2 → 🟢 Groq 
Request 3 → 🟢 Groq 
Request 4 → 🟢 Groq 
Request 5 → 🟢 Groq 


Observability — Log Every Single Call

In [49]:
import litellm
from litellm import completion

# A simple in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("❌ Call failed:", kwargs.get("exception"))
    # Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "krish"),
    ("Explain transformers.", "student_42"),
    ("What is fine-tuning?", "krish"),
]:
    completion(
        model="groq/openai/gpt-oss-120b",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

❌ Call failed: No module named 'fastapi'


ModuleNotFoundError: No module named 'fastapi'

In [51]:
import litellm
from litellm import completion

# A simple in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("❌ Call failed:", kwargs.get("exception"))

# Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "harsha"),
    ("Explain transformers.", "student_455"),
    ("What is fine-tuning?", "harsha1"),
]:
    completion(
        model="groq/openai/gpt-oss-120b",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

[
  {
    "model": "openai/gpt-oss-120b",
    "prompt": "What is RAG?",
    "input_tokens": 76,
    "output_tokens": 1585,
    "latency_sec": 3.83,
    "cost_usd": 0.0009624,
    "user": "harsha"
  },
  {
    "model": "openai/gpt-oss-120b",
    "prompt": "Explain transformers.",
    "input_tokens": 74,
    "output_tokens": 3072,
    "latency_sec": 7.0,
    "cost_usd": 0.0018543,
    "user": "student_455"
  }
]


Integrating the Gateway with LangChain

In [54]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
llm = ChatLiteLLM(model="groq/openai/gpt-oss-120b", temperature=0.3)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named harsha. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

- **Unified API layer** that routes requests from applications to one or multiple large language models (LLMs), handling authentication, rate‑limiting, and request formatting.  
- **Model‑agnostic orchestration** that lets you switch, combine, or cascade different LLM providers (e.g., OpenAI, Anthropic, local models) without changing client code.  
- **Operational safeguards** such as logging, content filtering, latency monitoring, and cost control, providing a single point for governance and observability.


A Real Example — Multi-Provider LangChain Chain with Fallbacks

In [55]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Primary model
primary = ChatLiteLLM(model="gpt-x")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="gpt-4o-mini", temperature=0.2)
fallback_2 = ChatLiteLLM(model="groq/openai/gpt-oss-120b", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)

❌ Call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-x
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
❌ Call failed: litellm.RateLimitError: RateLimitError: OpenAIException - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.
{
  "answer": "1. **Centralized access and management** – An LLM Gateway provides a single entry point to multiple language models, simplifying integration, version control, and monitoring across the organization.\n2. **Security, compliance, and governance** – It enforces authentication, authorization, data masking, and audit logging, ensuring that sensitive data is handled according to regulatory requirements.\n3. **Scalability and cost optimization** – By routing requests intelligently

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip
